# VAE Loss Function Experiments
**Tests:** LDS, FDS, Focal, BMSE, Alpha Recon, Gamma Contrast

### Kaggle Results Summary
| Loss Type | Best Config | Kaggle Score |
|-----------|-------------|--------------|
| Gamma Contrast | 0.0 | 0.26752 |
| Alpha Recon | 0.0 | 0.28368 |
| LDS | V1 | 0.28695 |
| Focal | 1.0 | 0.31616 |
| FDS | V4 | 0.44314 |

**Conclusion:** DIR losses don't help VAE. Best results with gamma=0 and alpha=0 (no reconstruction/contrast loss).

In [7]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')

# Path setup
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Models and utilities from project structure
from models.vae import SideslipVAE
from utils.data_loadingV2 import load_data
from utils.band_mapping import load_band_mapping, calculate_band_mae
from utils.simple_lds_loss import create_lds_weights
from utils.bmse_loss import BMCLoss
from utils.reproducibility_utils import seed_everything
from utils.visualization import plot_sweep_results, plot_method_comparison

SEED = 42
seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")

Device: cuda
Project root: /home/mdesouky/workspace/mait_g2_drift_challenge


In [8]:
# Load data
TRAIN_PATH = PROJECT_ROOT / "datasets_challenge" / "train"
COLUMNS_TO_DROP = ['sideSlip', 'vyCG', 'longitude', 'latitude', 'time']

train_loader, val_loader, scaler, _, _ = load_data(
    train_path=str(TRAIN_PATH), val_ratio=0.15, columns_to_drop=COLUMNS_TO_DROP,
    sequence_length=20, target="sideSlip", threshold=True, threshold_value=3.0,
    threshold_column="vxCG", batchsize=64, scaling=True, seed=SEED, shuffle=True
)
band_dict = load_band_mapping()

# Extract training targets for LDS weights
train_targets = []
for X, y in train_loader:
    train_targets.extend(y.numpy())
targets_flat = np.array(train_targets)
print(f"Train: {len(train_loader)} batches, Val: {len(val_loader)} batches")

V5 Data Loader: Stratified Temporal Split

Processing 22 files...
Total groups after threshold: 42

Train: Many=90.3%, Medium=7.6%, Few=2.1%
Val:   Many=96.6%, Medium=2.3%, Few=1.1%

Train: 120,445 sequences | Val: 16,635 sequences

Train: 1882 batches, Val: 260 batches


In [9]:
class VAEDIRLoss(nn.Module):
    """VAE loss with DIR components."""
    def __init__(self, beta_kl=0.005, dir_type='l1', weights=None):
        super().__init__()
        self.beta_kl, self.dir_type, self.lds_weights = beta_kl, dir_type, weights
        if dir_type == 'bmc': self.bmc_loss = BMCLoss(init_noise_sigma=1.0)
    
    def forward(self, outputs, x_input, y_true):
        y_pred, mu, logvar = outputs['y_pred'].squeeze(), outputs['mu'], outputs['logvar']
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        
        if self.dir_type == 'l1':
            pred_loss = F.l1_loss(y_pred, y_true)
        elif self.dir_type == 'weighted_l1':
            pred_loss = (torch.abs(y_pred - y_true) * (1.0 + torch.abs(y_true))).mean()
        elif self.dir_type == 'lds' and self.lds_weights is not None:
            indices = ((y_true + 5) / 10 * 100).long().clamp(0, len(self.lds_weights)-1)
            weights = torch.tensor([self.lds_weights[i] for i in indices]).to(device)
            pred_loss = (torch.abs(y_pred - y_true) * weights).mean()
        elif self.dir_type == 'focal':
            abs_diff = torch.abs(y_pred - y_true)
            pred_loss = (abs_diff * torch.pow(abs_diff + 0.1, 2.0)).mean()
        elif self.dir_type == 'bmc':
            pred_loss = self.bmc_loss(y_pred, y_true)
        else:
            pred_loss = F.l1_loss(y_pred, y_true)
        
        return pred_loss + self.beta_kl * kl_loss, {'pred': pred_loss.item(), 'kl': kl_loss.item()}

In [10]:
def train_vae_with_loss(loss_type, train_loader, val_loader, band_dict, lds_weights=None):
    """Train VAE with specific loss."""
    model = SideslipVAE(input_dim=22, hidden_dim=64, latent_dim=16, seq_len=20).to(device)
    loss_fn = VAEDIRLoss(beta_kl=0.005, dir_type=loss_type, weights=lds_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    
    best_mae, best_state = float('inf'), None
    for epoch in range(50):
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss, _ = loss_fn(model(X), X, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        
        if (epoch + 1) % 10 == 0:
            metrics = evaluate(model, val_loader, band_dict)
            if metrics['overall_mae'] < best_mae:
                best_mae = metrics['overall_mae']
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    if best_state: model.load_state_dict(best_state)
    return model, evaluate(model, val_loader, band_dict)

@torch.no_grad()
def evaluate(model, loader, band_dict):
    model.eval()
    preds, targets = [], []
    for X, y in loader:
        preds.extend(model(X.to(device))['y_pred'].cpu().numpy().flatten())
        targets.extend(y.numpy())
    return calculate_band_mae(np.array(preds), np.array(targets), band_dict)

## 1. Loss Function Comparison

### Kaggle Results - Focal Loss
| Gamma | Submission File | Kaggle Score |
|-------|-----------------|--------------|
| 0.5 | submission_Focal_0.5.csv | Error |
| **1.0** | submission_Focal_1.0.csv | **0.31616** |
| 3.0 | submission_Focal_3.0.csv | 0.39786 |

### Kaggle Results - Alpha Reconstruction
| Alpha | Submission File | Kaggle Score |
|-------|-----------------|--------------|
| **0.0** | submission_alpha_recon_0.0.csv | **0.28368** |
| 0.1 | submission_alpha_recon_0.1.csv | 0.33590 |
| 0.5 | submission_alpha_recon_0.5.csv | 0.31966 |
| 1.0 | submission_alpha_recon_1.0.csv | 0.29705 |
| 2.0 | submission_alpha_recon_2.0.csv | 0.34049 |

### Kaggle Results - Gamma Contrast
| Gamma | Submission File | Kaggle Score |
|-------|-----------------|--------------|
| **0.0** | submission_gamma_contrast_0.0.csv | **0.26752** |
| 0.1 | submission_gamma_contrast_0.1.csv | 0.31221 |
| 0.3 | submission_gamma_contrast_0.3.csv | 0.31762 |
| 0.5 | submission_gamma_contrast_0.5.csv | 0.28130 |
| 1.0 | submission_gamma_contrast_1.0.csv | 0.29306 |

**Best:** Both alpha_recon=0.0 and gamma_contrast=0.0 → reconstruction/contrast losses hurt prediction.

In [11]:
lds_weights = create_lds_weights(targets_flat, kernel='gaussian', ks=5, sigma=2)

loss_results = []
for loss_type in ['l1', 'weighted_l1', 'lds', 'focal', 'bmc']:
    print(f"Training {loss_type}...")
    weights = lds_weights if loss_type == 'lds' else None
    _, metrics = train_vae_with_loss(loss_type, train_loader, val_loader, band_dict, weights)
    print(f"  Overall={metrics['overall_mae']:.4f}, Few={metrics['few_mae']:.4f}")
    loss_results.append({'loss': loss_type, **metrics})

pd.DataFrame(loss_results)

Training l1...
  Overall=0.3147, Few=0.4509
Training weighted_l1...
  Overall=0.2577, Few=0.2185
Training lds...
  Overall=1.2752, Few=0.4009
Training focal...
  Overall=0.2928, Few=0.5672
Training bmc...
  Overall=0.3957, Few=0.7665


,loss,many_mae,many_count,many_pct,medium_mae,medium_count,medium_pct,few_mae,few_count,few_pct,overall_mae
0,l1,0.271827,4230,25.428314,0.489156,95,0.571085,0.450881,18,0.108206,0.314691
1,weighted_l1,0.241499,4230,25.428314,0.197777,95,0.571085,0.218481,18,0.108206,0.257689
2,lds,1.206367,4230,25.428314,1.197499,95,0.571085,0.400876,18,0.108206,1.275171
3,focal,0.269001,4230,25.428314,0.381604,95,0.571085,0.567242,18,0.108206,0.292806
4,bmc,0.358751,4230,25.428314,0.543485,95,0.571085,0.766519,18,0.108206,0.395694


## 2. LDS Kernel Sweep

### Kaggle Results - LDS Variants
| Version | Submission File | Kaggle Score |
|---------|-----------------|--------------|
| **V1** | predictions_LDS_V1.csv | **0.28695** |
| V2 | predictions_LDS_V2.csv | 0.36409 |
| V3 | predictions_LDS_V3.csv | 0.45150 |
| V4 | predictions_LDS_V4.csv | 0.32742 |

### Kaggle Results - FDS Variants
| Version | Submission File | Kaggle Score |
|---------|-----------------|--------------|
| V1 | predictions_FDS_V1.csv | 0.61806 |
| V2 | predictions_FDS_V2.csv | 0.48196 |
| V3 | predictions_FDS_V3.csv | 0.62249 |
| **V4** | predictions_FDS_V4.csv | **0.44314** |

### Kaggle Results - FDS/BMSE (Error Submissions)
| Config | Submission File | Status |
|--------|-----------------|--------|
| FDS 1.0 | submission_FDS_1.0.csv | Error |
| FDS 2.0 | submission_FDS_2.0.csv | Error |
| FDS 3.0 | submission_FDS_3.0.csv | Error |
| BMSE K3 | submission_BMSE_K3.csv | Error |
| BMSE K5 | submission_BMSE_K5.csv | Error |
| LDS 1.0 sqrt_inv | submission_LDS_1.0_sqrt_inv.csv | Error |
| LDS 2.0 inv | submission_LDS_2.0_inv.csv | Error |
| LDS 3.0 sqrt_inv | submission_LDS_3.0_sqrt_inv.csv | Error |

**Conclusion:** LDS V1 is best (0.28695), but still worse than gamma=0 baseline (0.26752). FDS consistently degrades performance (0.44-0.62). BMSE failed.

In [12]:
lds_results = []
kernels = [('gaussian', 3, 1), ('gaussian', 5, 2), ('laplacian', 5, 2), ('triang', 5, None)]

for kernel, ks, sigma in kernels:
    config_name = f"{kernel}_ks{ks}_s{sigma}"
    print(f"Training {config_name}...")
    weights = create_lds_weights(targets_flat, kernel=kernel, ks=ks, sigma=sigma)
    _, metrics = train_vae_with_loss('lds', train_loader, val_loader, band_dict, weights)
    print(f"  Overall={metrics['overall_mae']:.4f}")
    lds_results.append({'config': config_name, **metrics})

pd.DataFrame(lds_results)

Training gaussian_ks3_s1...
  Overall=3.9185
Training gaussian_ks5_s2...
  Overall=1.3611
Training laplacian_ks5_s2...
  Overall=1.3738
Training triang_ks5_sNone...
  Overall=1.3768


,config,many_mae,many_count,many_pct,medium_mae,medium_count,medium_pct,few_mae,few_count,few_pct,overall_mae
0,gaussian_ks3_s1,3.763412,4230,25.428314,6.993185,95,0.571085,9.277526,18,0.108206,3.918475
1,gaussian_ks5_s2,1.326881,4230,25.428314,1.347035,95,0.571085,0.358893,18,0.108206,1.361069
2,laplacian_ks5_s2,1.371398,4230,25.428314,1.085164,95,0.571085,0.530151,18,0.108206,1.373840
3,triang_ks5_sNone,1.366305,4230,25.428314,1.132779,95,0.571085,0.450861,18,0.108206,1.376775


## 3. Ensemble Experiment

### Kaggle Results - VAE Ensembles
| Ensemble Type | Submission File | Kaggle Score |
|---------------|-----------------|--------------|
| **Ensemble V4** | submission_ensemble_v4.csv | **0.26493** |
| Ensemble Optimal | submission_ensemble_optimal.csv | 0.26692 |
| Ensemble Top3 | submission_ensemble_top3_overall.csv | 0.27775 |
| Ensemble Weighted | submission_ensemble_weighted.csv | 0.36272 |
| Ensemble All | submission_ensemble_all.csv | 0.38072 |

**Conclusion:** Best ensemble (V4) achieves 0.26493, marginally better than single gamma=0 model (0.26752). Large ensembles hurt.

In [13]:
models, all_preds = [], []
for i, loss_type in enumerate(['l1', 'weighted_l1', 'focal']):
    seed_everything(SEED + i)
    print(f"Training ensemble member: {loss_type}...")
    model, _ = train_vae_with_loss(loss_type, train_loader, val_loader, band_dict)
    models.append(model)
    
    model.eval()
    preds = []
    for X, _ in val_loader:
        with torch.no_grad():
            preds.extend(model(X.to(device))['y_pred'].cpu().numpy().flatten())
    all_preds.append(np.array(preds))

val_targets = []
for _, y in val_loader:
    val_targets.extend(y.numpy())
val_targets = np.array(val_targets)

# Ensemble predictions
mean_preds = np.mean(all_preds, axis=0)
weighted_preds = 0.4*all_preds[0] + 0.35*all_preds[1] + 0.25*all_preds[2]

mean_metrics = calculate_band_mae(mean_preds, val_targets, band_dict)
weighted_metrics = calculate_band_mae(weighted_preds, val_targets, band_dict)

print(f"\nMean ensemble: {mean_metrics['overall_mae']:.4f}")
print(f"Weighted ensemble: {weighted_metrics['overall_mae']:.4f}")

Training ensemble member: l1...
Training ensemble member: weighted_l1...
Training ensemble member: focal...

Mean ensemble: 0.2126
Weighted ensemble: 0.2160


## Summary

### All VAE Loss Experiments - Kaggle Results
| Category | Config | Submission File | Kaggle Score |
|----------|--------|-----------------|--------------|
| **Beta** | **0.005** | submission_beta_0.005.csv | **0.25319** |
| Gamma | 0.0 | submission_gamma_contrast_0.0.csv | 0.26752 |
| Ensemble | V4 | submission_ensemble_v4.csv | 0.26493 |
| Ensemble | Optimal | submission_ensemble_optimal.csv | 0.26692 |
| Arch | Deep | submission_arch_deep.csv | 0.26613 |
| Arch | Deep 2L-128 | submission_deep_2layer_128.csv | 0.26502 |
| Alpha | 0.0 | submission_alpha_recon_0.0.csv | 0.28368 |
| LDS | V1 | predictions_LDS_V1.csv | 0.28695 |
| Gamma | 0.5 | submission_gamma_contrast_0.5.csv | 0.28130 |
| Focal | 1.0 | submission_Focal_1.0.csv | 0.31616 |
| LDS | V4 | predictions_LDS_V4.csv | 0.32742 |
| LDS | V2 | predictions_LDS_V2.csv | 0.36409 |
| Focal | 3.0 | submission_Focal_3.0.csv | 0.39786 |
| FDS | V4 | predictions_FDS_V4.csv | 0.44314 |
| LDS | V3 | predictions_LDS_V3.csv | 0.45150 |
| FDS | V2 | predictions_FDS_V2.csv | 0.48196 |
| FDS | V1 | predictions_FDS_V1.csv | 0.61806 |
| FDS | V3 | predictions_FDS_V3.csv | 0.62249 |

### Key Findings
| Finding | Result |
|---------|--------|
| Best VAE Single | beta=0.005 → **0.25319** |
| Best Ensemble | V4 → 0.26493 |
| Best DIR (LDS) | V1 → 0.28695 |
| Worst DIR (FDS) | V3 → 0.62249 |
| Baseline RNN | **0.217** |

### Conclusions
1. **DIR losses hurt VAE:** LDS, FDS, Focal all degrade performance
2. **Reconstruction loss hurts:** alpha_recon=0 is optimal
3. **Contrast loss hurts:** gamma_contrast=0 is optimal
4. **Simple is better:** Best VAE (0.25319) uses just prediction loss + minimal KL
5. **VAE still loses to RNN:** +16.7% error vs baseline RNN (0.217)

## 4. Predictions vs Ground Truth

In [14]:
# Using utils/visualization.py
from utils.visualization import plot_predictions_scatter

best_preds = all_preds[1]  # weighted_l1 model


# Loss function comparison
loss_df = pd.DataFrame(loss_results)
fig_loss = plot_method_comparison(
    [{'method': r['loss'], 'score': r['overall_mae']} for r in loss_results],
    title='Loss Function Comparison: Overall MAE'
)
fig_loss.show()

## Interactive Predictions vs Ground Truth

In [15]:
# Interactive line plot showing predictions overlaid on ground truth
from utils.visualization import plot_predictions_vs_truth

# Using best model predictions (weighted_l1)
fig_line = plot_predictions_vs_truth(
    best_preds, val_targets,
    title='VAE (WeightedL1): Predictions vs Ground Truth (Interactive)',
    max_samples=5000
)
fig_line.show()